# ICCP 2026 Summer School: Cryo-EM Image Formation and Homogeneous Reconstruction

## Student Setup
%% TODO: the data should all be downloaded from a single !wget command by pushing all necessary data to a github folder
%% No zipping nothing, new github folder should be like cryoem-homomgeneous-iccp-2026 with all data (even this notebook (?))

%% Also in TODO: in code given in (recovar), take out any functions code that are not used, I want as little code as possible in there 

1. Install UCSF ChimeraX from the [official download page](https://www.rbvi.ucsf.edu/chimerax/download.html).
2. Open `6vxx.cif` in ChimeraX. This is a SARS-CoV-2 spike protein model from [PDB 6VXX](https://www.rcsb.org/structure/6VXX). The file is included with the notebook; the notebook downloads it if needed.
3. In Colab, run the first cell. It downloads the small helper code and data files from GitHub with one `wget` command.
4. Later in the notebook, download the `.mrc` volumes you generate and open them in ChimeraX too.


In [ ]:
!wget -q -O iccp2026_cryoem_data.zip https://github.com/ma-gilles/cryoem-homogeneous-iccp-2026/releases/download/v0.1/iccp2026_cryoem_data.zip
!unzip -q -o iccp2026_cryoem_data.zip


In [ ]:
%pip install -q -r requirements.txt


In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np

import cryoem

GRID_SIZE = 128
BOX_SIZE_ANGSTROM = 300
N_SYNTHETIC_IMAGES = 96
BATCH_SIZE = 24
rng = np.random.default_rng(0)

plt.rcParams["figure.dpi"] = 120


In [ ]:
PDB_PATH = cryoem.get_pdb("6VXX", "6vxx.cif")

## Scattering Potential

What is observed in cryo-EM is the Coulomb scattering potential of the molecule. We write it as a function

$$V:\mathbb{R}^3 \to \mathbb{R}.$$

For each spatial location $x=(x_1,x_2,x_3)$, the value $V(x)$ describes how strongly the molecule scatters the electron wave near that location. An atomic model gives atom positions and element types. The next cell places those atoms on a regular 3D grid, producing a volume that we can save as an MRC file.

In [ ]:
VOLUME_MRC = Path(f"6vxx_scattering_{GRID_SIZE}.mrc")
volume, VOXEL_SIZE = cryoem.make_scattering_volume(PDB_PATH, GRID_SIZE, BOX_SIZE_ANGSTROM, VOLUME_MRC)
cryoem.show_slices(volume, "Scattering potential from 6VXX")

print(f"Saved {VOLUME_MRC}. Download it and open it in ChimeraX.")

You have generated the scattering potential on a grid. Download `6vxx_scattering_128.mrc` and open it in ChimeraX.

Move the threshold up and down and compare to the atomic model.

## Forward Model: From a Volume to One Image

A simplified cryo-EM image formation model is

$$y_i = h_i * P_{R_i}V + \epsilon_i, \quad i = 1,\ldots,n.$$

%% TODO: clean up make sure everything is properly defined 
Here $y_i$ is the observed particle image, $R_i \in SO(3)$ is the particle orientation, $P_{R_i}$ is the line-integral projection operator, $h_i$ is the microscope point-spread function (PSF), $*$ denotes convolution in the image plane, and $\epsilon_i$ is noise.

In continuous notation,

$$(P_R V)(x_1,x_2) = \int_{-\infty}^{\infty} V\!\left(R^T(x_1,x_2,x_3)^T\right)\,dx_3.$$

For the axis-aligned case $R=I$, the grid approximation is just a sum along one axis. The cell below computes this projection, applies a simple CTF in Fourier space, displays the corresponding PSF in real space, and then adds noise.


In [ ]:
projection = volume.sum(axis=0)
projection = projection / np.max(np.abs(projection))

## TODO: ADD A PLOT OF THE PSF (IFT OF CTF), AND SHOW PROJ IMAGE  * PSF + NOISE = TRUE IMAGE (ALL AS INDEP IMAGES), ONE PROJ IMAGE, ONE PSF, ONE NOISE, AND THE TOTAL

projection_ft = cryoem.dft2(projection)
ctf = cryoem.simple_ctf(GRID_SIZE, VOXEL_SIZE, defocus_um=1.8)
psf = np.fft.ifftshift(np.fft.ifft2(np.fft.ifftshift(ctf))).real
psf = psf / np.max(np.abs(psf))
ctf_projection = np.fft.ifftshift(np.fft.ifft2(np.fft.ifftshift(projection_ft * ctf))).real
noise = rng.normal(scale=0.25 * np.std(ctf_projection), size=ctf_projection.shape)
noisy_image = ctf_projection + noise

cryoem.show_images(
    [projection, psf, ctf_projection, noise, noisy_image],
    ["projection", "PSF", "projection convolved with PSF", "noise", "image"],
    columns=5,
)


## Fourier Slice Theorem

The most important mathematical tool in tomographic reconstruction is the Fourier slice theorem. With the projection operator above, it says

$$\widehat{P_R V}(\omega_1,\omega_2) = \widehat V\!\left(R^T(\omega_1,\omega_2,0)^T\right).$$

That is, the 2D Fourier transform of a projection image is a central plane through the 3D Fourier transform of the volume. The plane orientation is determined by the particle orientation $R$.

For $R=I$, this reduces to

$$\widehat{P_I V}(\omega_1,\omega_2)=\widehat V(\omega_1,\omega_2,0).$$

The next cell checks the discrete version on our grid.
%% TODO: (instructions for students to include): BELOW, YOU ARE TASKED TO use the fourier slice theorem to compute the projection along the x-axis (axis = 0 ), and z -axis (axis =2) using the Fourier slice theorem, and compare it to the naive implementaiton (i.e. np.sum(V, x = dim)). Make sure they are the same!
%% An important thing: As computed, below the Fourier grid is arranged as follows, for a function defiend on N pixels with N even,
%% - N//2 , -N//2 -1 .... 0 , N ... 


In [ ]:
def dft3(vol):
    return np.fft.fftshift(np.fft.fftn(np.fft.fftshift(vol)))


def idft3(vol_ft):
    return np.fft.ifftshift(np.fft.ifftn(np.fft.ifftshift(vol_ft)))


def idft2(image_ft):
    return np.fft.ifftshift(np.fft.ifft2(np.fft.ifftshift(image_ft)))


def project_along_axis_from_fourier_slice(vol, axis):
    # %% Given the 3D volume (in real space), compute the projection along the requested axis using the Fourier slice theorem. The function returns the 2D projection image in real space.
    # %% complete this function!
    vol_ft = dft3(vol)
    central_slice_ft = np.take(vol_ft, vol.shape[axis] // 2, axis=axis)
    return idft2(central_slice_ft).real


# %% TODO Complete this piece of code for students:
volume_ft = dft3(volume).astype(np.complex64)

vol_proj_x_ref = np.sum(volume, axis=0)
vol_proj_x_fst = project_along_axis_from_fourier_slice(volume, axis=0)
vol_proj_z_ref = np.sum(volume, axis=2)
vol_proj_z_fst = project_along_axis_from_fourier_slice(volume, axis=2)

x_relative_error = np.linalg.norm(vol_proj_x_ref - vol_proj_x_fst) / np.linalg.norm(vol_proj_x_ref)
z_relative_error = np.linalg.norm(vol_proj_z_ref - vol_proj_z_fst) / np.linalg.norm(vol_proj_z_ref)
print(f"x-axis projection relative error: {x_relative_error:.3e}")
print(f"z-axis projection relative error: {z_relative_error:.3e}")

projection_from_real_space = vol_proj_x_ref
projection_from_fourier_slice = vol_proj_x_fst

cryoem.show_images(
    [vol_proj_x_ref, vol_proj_x_fst, vol_proj_x_ref - vol_proj_x_fst, vol_proj_z_ref, vol_proj_z_fst, vol_proj_z_ref - vol_proj_z_fst],
    ["x-axis sum", "x-axis Fourier slice", "x difference", "z-axis sum", "z-axis Fourier slice", "z difference"],
    columns=3,
)


## Non-Axis-Aligned Views

When the direction of propagation is not axis-aligned, the Fourier slice does not land exactly on grid points. We need interpolation in the 3D Fourier volume. The next cell extracts a central slice at a 45 degree viewing angle.

%% TODO add a comment: furthermore, the convolution with point spread function is simply multiplication in the Fourier domain. Thus, both operations are very easy to implement in the Foureir domain, and that is why we typically work in the Fourier domain when doing reconstruction.

Furthermore, convolution with the point-spread function is multiplication by the CTF in the Fourier domain. Projection and CTF application are therefore both simple Fourier-domain operations, which is why reconstruction algorithms usually work directly with Fourier coefficients.


In [ ]:
# %% TODO: make a little example. (instructions for students to write) If axis aligned (rotation = 0), then we get the same thing as above,
# %% However, we can now also project along non-axis aligned, ehre's an example:

axis_aligned_slice_ft = cryoem.slice(volume_ft, np.eye(3, dtype=np.float32))
axis_aligned_projection = idft2(axis_aligned_slice_ft).real.T
axis_aligned_reference = project_along_axis_from_fourier_slice(volume, axis=2)
axis_aligned_error = np.linalg.norm(axis_aligned_reference - axis_aligned_projection) / np.linalg.norm(axis_aligned_reference)
print(f"axis-aligned interpolation relative error: {axis_aligned_error:.3e}")

rotation_45 = cryoem.rotation_y(45)
rotated_slice_ft = cryoem.slice(volume_ft, rotation_45)
rotated_projection = idft2(rotated_slice_ft).real.T

cryoem.show_images(
    [axis_aligned_reference, axis_aligned_projection, rotated_projection],
    ["axis-aligned Fourier slice", "axis-aligned interpolator", "45 degree view"],
    columns=3,
)


## Synthetic Projection Dataset

Now we generate a small synthetic cryo-EM dataset: many random viewing directions, one CTF per image, and additive noise. This is the dataset we will reconstruct from.

In [ ]:
noisy_images, rotations, ctfs, clean_slices_ft = cryoem.make_synthetic_dataset(
    volume_ft, N_SYNTHETIC_IMAGES, VOXEL_SIZE, seed=2
)

cryoem.show_images(noisy_images[:10], columns=5, title=f"{N_SYNTHETIC_IMAGES} synthetic noisy projections")

## Homogeneous Reconstruction with Known Poses

In real reconstructions, the particle orientations $R_i$ are not known and must be inferred. Here we use the known synthetic orientations and focus only on the inversion step.

Ignoring pose estimation, a homogeneous reconstruction can be written as

$$\min_V \sum_i \| y_i - C_i P_{R_i} V\|_2^2.$$

The normal equations are:

$$\left(\sum_i P_{R_i}^T C_i^T C_i P_{R_i}\right)V = \sum_i P_{R_i}^T C_i^T y_i.$$

<!-- In Fourier space, our gridded implementation approximates this by accumulating a numerator and a denominator: -->

$$\widehat V \approx \left(\sum_i P_{R_i}^T C_i^T C_i P_{R_i}\right)^{-1} \left(\sum_i P_{R_i}^T C_i^T \widehat y_i \right)$$

%% TODO: Thanks to the nice structure in the Fourier domain, this can be rewritten as :

Thanks to the Fourier-domain structure, our gridded implementation approximates this by accumulating a numerator and a denominator at each 3D Fourier voxel:

$$\widehat V \approx \frac{\sum_i P_{R_i}^T C_i^T \widehat y_i}{\sum_i P_{R_i}^T c_i^2}.$$

where $c_i \in \mathbb{R}^{d^2}$ is the CTF function in the Fourier domain. In other words, to reconstruct from images, we take a weighted linear combination of backprojected images and divide by the backprojected weights.

The key operation is $P_{R_i}^T$: take a 2D Fourier image and insert it back into the corresponding central plane of a 3D Fourier volume.


In [ ]:
one_slice_ft = clean_slices_ft[0]
one_backprojection = cryoem.backproject(one_slice_ft, rotations[0], GRID_SIZE)

energy = np.abs(one_backprojection)
cryoem.show_slices(energy / energy.max(), "One backprojected Fourier slice", percentile=99)



A single image only populates one central plane. A reconstruction needs many images, each filling a different plane. The next cell builds the numerator and denominator of the normal-equation approximation by summing over the synthetic images.

In [ ]:
# %% TODO: another cell with say 3 slices 

slice_indices = [GRID_SIZE // 2 - 24, GRID_SIZE // 2, GRID_SIZE // 2 + 24]
cryoem.show_images(
    [energy[i] for i in slice_indices],
    [f"Fourier z-slice {i}" for i in slice_indices],
    columns=3,
    cmap="magma",
    title="Three slices through one backprojected image",
)


In [ ]:
rhs = np.zeros((GRID_SIZE, GRID_SIZE, GRID_SIZE), dtype=np.complex64)
weights = np.zeros((GRID_SIZE, GRID_SIZE, GRID_SIZE), dtype=np.float32)

for start in range(0, N_SYNTHETIC_IMAGES, BATCH_SIZE):
    stop = min(start + BATCH_SIZE, N_SYNTHETIC_IMAGES)
    image_ft = np.stack([cryoem.dft2(image) for image in noisy_images[start:stop]]).astype(np.complex64)
    ctf_batch = ctfs[start:stop]

    rhs += cryoem.backproject(ctf_batch * image_ft, rotations[start:stop], GRID_SIZE)
    weights += cryoem.backproject(ctf_batch**2, rotations[start:stop], GRID_SIZE).real

scale = np.percentile(weights[weights > 0], 95)
recon_ft = rhs / (weights + 1e-6 * scale)
recon_unregularized = idft3(recon_ft).real.astype(np.float32)

cryoem.save_mrc("reconstruction_unregularized.mrc", recon_unregularized, VOXEL_SIZE)
cryoem.show_slices(recon_unregularized, "Unregularized reconstruction")


This is a reconstruction, but it is noisy. The division by small values in the denominator amplifies frequencies that were weakly observed or lie near CTF zeros. A standard fix is to regularize the division:

$$\widehat V_\lambda = \frac{\sum_i P_{R_i}^T C_i^T \widehat y_i}{\sum_i P_{R_i}^T C_i^T C_i P_{R_i}+\lambda}.$$

Try changing the regularization values below. Small values keep more detail but amplify noise. Large values suppress noise but blur the map.

In [ ]:
regularization_fractions = [0.002, 0.01, 0.05, 0.2]
recons = []

for frac in regularization_fractions:
    recon_ft = rhs / (weights + frac * scale)
    recon = idft3(recon_ft).real.astype(np.float32)
    recons.append(recon)
    cryoem.save_mrc(f"reconstruction_reg_{frac:g}.mrc", recon, VOXEL_SIZE)
    print(f"saved reconstruction_reg_{frac:g}.mrc")

cryoem.show_images(
    [r[GRID_SIZE // 2] for r in recons],
    [f"lambda frac {f:g}" for f in regularization_fractions],
    columns=len(recons),
    cmap="magma",
)


Download one or more of the reconstructed `.mrc` files and open them in ChimeraX. Compare them to `6vxx_scattering_128.mrc`.

Questions:

1. What features are stable as you change $\lambda$? Which $\lambda$ is better? Can you come up with a better way to regularize the problem that a constant $\lambda$ ?


## Real Particle Images and Reconstruction: EMPIAR-10028 Subsampled Dataset

The folder includes a 1000-particle subsampled dataset derived from **EMPIAR-10028**:

```text
particles_1000_128.mrcs
particles_1000_128.star
emd_2660.map.gz
```

The STAR file contains CTF parameters, projection directions, and in-plane shifts from a previous refinement. We are not estimating poses here; we are using known poses to run the same regularized backprojection reconstruction on real images.

Change `real_regularization` and rerun the cell. For what values does the map start to look reasonable in ChimeraX?

In [ ]:
real_images, real_ctf_params, real_voxel_size, pose_status, real_rotations, real_translations_pixels = cryoem.load_empiar10028_particles()

indices = np.linspace(0, len(real_images) - 1, 12, dtype=int)
cryoem.show_images(real_images[indices], [f"particle {i}" for i in indices], columns=4, title="Real EMPIAR-10028 particles")

real_ctfs = cryoem.ctfs(real_ctf_params, GRID_SIZE, real_voxel_size)
cryoem.show_images([real_images[0], real_ctfs[0]], ["real particle", "CTF from metadata"], columns=2)

images_for_reconstruction = real_images - real_images.mean(axis=(1, 2), keepdims=True)
images_for_reconstruction = images_for_reconstruction / images_for_reconstruction.std(axis=(1, 2), keepdims=True)

rhs = np.zeros((GRID_SIZE, GRID_SIZE, GRID_SIZE), dtype=np.complex64)
weights = np.zeros((GRID_SIZE, GRID_SIZE, GRID_SIZE), dtype=np.float32)
REAL_BATCH_SIZE = 100

for start in range(0, len(images_for_reconstruction), REAL_BATCH_SIZE):
    stop = start + REAL_BATCH_SIZE
    image_ft = np.stack([cryoem.dft2(image) for image in images_for_reconstruction[start:stop]])
    image_ft = cryoem.translate(image_ft, real_translations_pixels[start:stop], GRID_SIZE)
    rhs += cryoem.backproject(real_ctfs[start:stop] * image_ft, real_rotations[start:stop], GRID_SIZE)
    weights += cryoem.backproject(real_ctfs[start:stop] ** 2, real_rotations[start:stop], GRID_SIZE).real

real_regularization = 0.05
lam = real_regularization * np.percentile(weights[weights > 0], 95)
real_reconstruction = idft3(rhs / (weights + lam)).real.astype(np.float32)
cryoem.save_mrc("empiar10028_reconstruction_1000_live_128.mrc", real_reconstruction, real_voxel_size)
cryoem.show_slices(real_reconstruction, f"Live reconstruction from 1000 real particles, regularization={real_regularization}")

published_reconstruction, published_voxel_size = cryoem.load_empiar10028_reference_map(out_size=GRID_SIZE)
cryoem.show_slices(published_reconstruction, "Published EMPIAR-10028 reconstruction, downsampled")

print("Saved empiar10028_reconstruction_1000_live_128.mrc")
print("Saved empiar10028_published_reconstruction_128.mrc")
print(f"Pose metadata status in the 1000-particle subset: {pose_status}")